# CSE534 Updated Multi-Persona Retraining Notebook

This notebook retrains the updated model on the **combined all-persona dataset**:

- universal tool semantics: reminder/calendar/alarm/notes and create/update/delete
- persona-specific importance: Arjun, Dev, Margaret, Neel
- confusing boundary examples: vague, unconfirmed, or obsolete plans

Recommended runtime: **Colab GPU**. T4 works; L4/A100 is faster.


## 1. Check GPU

If this does not show a GPU, change Colab runtime to `Runtime -> Change runtime type -> GPU`.


In [ ]:
!nvidia-smi


## 2. Mount Google Drive And Set Project Folder

Put your project files in the Drive folder below, or change `PROJECT_DIR` to wherever you uploaded them.

Required files for baseline + context + state/context SFT:

- `train_sft.py`
- `train_sft_ctx2.py`
- `train_sft_state_ctx2.py`
- `eval.py`
- `json_decision.py`
- `all_sft.jsonl`, `all_test.jsonl`, `all_stats.json`
- `all_ctx2_sft.jsonl`, `all_ctx2_test.jsonl`, `all_ctx2_stats.json` if you do not regenerate in Colab
- `all_state_ctx2_sft.jsonl`, `all_state_ctx2_test.jsonl`, `all_state_ctx2_stats.json` if you do not regenerate in Colab

If `REGENERATE_JSONL=True`, also upload `parse_json_dataset.py`, `build_combined_persona_dataset.py`, and the four source persona JSON files.


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/CSE534')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
%cd {PROJECT_DIR}
!ls -lh


## 3. Optional: Upload Files Directly Instead Of Drive

Run this only if you did not place the files in Drive. Upload the required `.py` and `.jsonl` files when prompted.


In [ ]:
# Optional direct upload
# from google.colab import files
# uploaded = files.upload()
# !ls -lh


## 4. Prepare Or Verify Dataset Files

This notebook compares a current-utterance baseline against a previous-2-turn context model.

Use the default mode when you already uploaded/generated the `all_*.jsonl` and `all_*_stats.json` files.

Set `REGENERATE_JSONL = True` only if you uploaded the four source persona JSON files and want Colab to rebuild baseline, context, and state+context JSONL files before training.


In [ ]:
from pathlib import Path
import json

# False = use uploaded all_*.jsonl, all_ctx2_*.jsonl, and all_state_ctx2_*.jsonl files.
# True = regenerate all JSONL files from the four persona source JSON files inside Colab.
REGENERATE_JSONL = True
RUN_CONTEXT_EXPERIMENT = True
RUN_STATE_CONTEXT_EXPERIMENT = True
CONTEXT_TURNS = 2
CONTEXT_SUFFIX = 'ctx2'
STATE_CONTEXT_SUFFIX = 'state_ctx2'
HOLDOUTS = [9, 10, 16, 17]
HOLDOUT_ARG = " ".join(str(x) for x in HOLDOUTS)

base_required = [
    'train_sft.py',
    'train_sft_ctx2.py',
    'train_sft_state_ctx2.py',
    'eval.py',
    'json_decision.py',
]

source_files = {
    'arjun': 'arjun_10_conversations_evolving_tool_calls_labeled.json',
    'dev': 'dev_entrepreneur_10_conversations_evolving_tool_calls_labeled.json',
    'margaret': 'margaret_10_conversations_evolving_tool_calls_labeled.json',
    'neel': 'neel_neuroscience_10_conversations_evolving_tool_calls_labeled.json',
}

baseline_datasets = ['all_sft.jsonl', 'all_test.jsonl', 'all_stats.json']
context_datasets = [
    f'all_{CONTEXT_SUFFIX}_sft.jsonl',
    f'all_{CONTEXT_SUFFIX}_test.jsonl',
    f'all_{CONTEXT_SUFFIX}_stats.json',
]
state_context_datasets = [
    f'all_{STATE_CONTEXT_SUFFIX}_sft.jsonl',
    f'all_{STATE_CONTEXT_SUFFIX}_test.jsonl',
    f'all_{STATE_CONTEXT_SUFFIX}_stats.json',
]

if REGENERATE_JSONL:
    required = base_required + [
        'parse_json_dataset.py',
        'build_combined_persona_dataset.py',
    ] + list(source_files.values())
else:
    required = base_required + baseline_datasets
    if RUN_CONTEXT_EXPERIMENT:
        required += context_datasets
    if RUN_STATE_CONTEXT_EXPERIMENT:
        required += state_context_datasets

missing = [f for f in required if not Path(f).exists()]
if missing:
    raise FileNotFoundError('Missing required files: ' + ', '.join(missing))

if REGENERATE_JSONL:
    for persona, source_path in source_files.items():
        print(f'Regenerating baseline {persona} JSONL files from {source_path}...')
        !python3 parse_json_dataset.py {source_path} {persona} --holdout {HOLDOUT_ARG}

        if RUN_CONTEXT_EXPERIMENT:
            print(f'Regenerating context {persona} JSONL files from {source_path}...')
            !python3 parse_json_dataset.py {source_path} {persona} --holdout {HOLDOUT_ARG} --context-turns {CONTEXT_TURNS} --suffix {CONTEXT_SUFFIX}

        if RUN_STATE_CONTEXT_EXPERIMENT:
            print(f'Regenerating state + context {persona} JSONL files from {source_path}...')
            !python3 parse_json_dataset.py {source_path} {persona} --holdout {HOLDOUT_ARG} --context-turns {CONTEXT_TURNS} --include-state --suffix {STATE_CONTEXT_SUFFIX}

    print('Building combined all-persona baseline JSONL files...')
    !python3 build_combined_persona_dataset.py

    if RUN_CONTEXT_EXPERIMENT:
        print('Building combined all-persona context JSONL files...')
        !python3 build_combined_persona_dataset.py --suffix {CONTEXT_SUFFIX}

    if RUN_STATE_CONTEXT_EXPERIMENT:
        print('Building combined all-persona state + context JSONL files...')
        !python3 build_combined_persona_dataset.py --suffix {STATE_CONTEXT_SUFFIX}

expected_datasets = baseline_datasets
if RUN_CONTEXT_EXPERIMENT:
    expected_datasets += context_datasets
if RUN_STATE_CONTEXT_EXPERIMENT:
    expected_datasets += state_context_datasets

for dataset_path in expected_datasets:
    if not Path(dataset_path).exists():
        raise FileNotFoundError(f'Expected dataset file was not created: {dataset_path}')

def count_jsonl(path):
    return sum(1 for _ in open(path))

print('HOLDOUTS =', HOLDOUTS)
for dataset_path in expected_datasets:
    print(dataset_path, count_jsonl(dataset_path), 'rows')

sample = json.loads(open('all_sft.jsonl').readline())
print('''
Baseline user preview:
''', sample['messages'][1]['content'][:1200])
if RUN_CONTEXT_EXPERIMENT:
    ctx_sample = json.loads(open(f'all_{CONTEXT_SUFFIX}_sft.jsonl').readline())
    print('''
Context user preview:
''', ctx_sample['messages'][1]['content'][:1200])
if RUN_STATE_CONTEXT_EXPERIMENT:
    state_sample = json.loads(open(f'all_{STATE_CONTEXT_SUFFIX}_sft.jsonl').readline())
    print('''
State + context user preview:
''', state_sample['messages'][1]['content'][:1200])


## 5. Install Dependencies

After this cell finishes, Colab may ask you to restart the session. If it does, restart and continue from the next cell.


In [ ]:
!pip install -U pip
!pip install -U "unsloth[colab-new]" trl transformers datasets accelerate peft bitsandbytes sentencepiece protobuf


## 6. Record Dataset Snapshot For Comparison

This saves label distributions for both the current-utterance baseline and the ctx2 context dataset.


In [ ]:
import json, time
from collections import Counter
from pathlib import Path

RUN_ID = time.strftime('%Y%m%d_%H%M%S')
RESULTS_DIR = Path(f'results_all_{RUN_ID}')
RESULTS_DIR.mkdir(exist_ok=True)

def load_jsonl(path):
    return [json.loads(line) for line in open(path) if line.strip()]

def label_counts(rows):
    tool_counts = Counter()
    importance_counts = Counter()
    persona_counts = Counter()
    for row in rows:
        target = json.loads(row['messages'][2]['content'])
        tool_counts[target['tool'] or 'None'] += 1
        importance_counts[target['importance']] += 1
        persona_counts[row.get('persona', 'unknown')] += 1
    return dict(tool_counts), dict(importance_counts), dict(persona_counts)

sft_rows = load_jsonl('all_sft.jsonl')
test_rows = load_jsonl('all_test.jsonl')
tool_counts, importance_counts, persona_counts = label_counts(sft_rows)

snapshot = {
    'run_id': RUN_ID,
    'baseline_sft_rows': len(sft_rows),
    'baseline_test_rows': len(test_rows),
    'baseline_tool_counts_sft': tool_counts,
    'baseline_importance_counts_sft': importance_counts,
    'baseline_persona_counts_sft': persona_counts,
    'context_experiment': RUN_CONTEXT_EXPERIMENT,
    'context_turns': CONTEXT_TURNS if RUN_CONTEXT_EXPERIMENT else 0,
}

if RUN_CONTEXT_EXPERIMENT:
    ctx_sft_rows = load_jsonl(f'all_{CONTEXT_SUFFIX}_sft.jsonl')
    ctx_test_rows = load_jsonl(f'all_{CONTEXT_SUFFIX}_test.jsonl')
    ctx_tool_counts, ctx_importance_counts, ctx_persona_counts = label_counts(ctx_sft_rows)
    snapshot.update({
        'context_suffix': CONTEXT_SUFFIX,
        'context_sft_rows': len(ctx_sft_rows),
        'context_test_rows': len(ctx_test_rows),
        'context_tool_counts_sft': ctx_tool_counts,
        'context_importance_counts_sft': ctx_importance_counts,
        'context_persona_counts_sft': ctx_persona_counts,
    })

Path(RESULTS_DIR / 'dataset_snapshot.json').write_text(json.dumps(snapshot, indent=2))
print(json.dumps(snapshot, indent=2))


## 7. Train SFT Baseline And Context Model

This creates `all-sft-merged/` and, when `RUN_CONTEXT_EXPERIMENT=True`, `all-ctx2-sft-merged/`.


In [ ]:
import os, time
os.environ['PERSONA'] = 'all'

start = time.time()
!PERSONA=all python3 train_sft.py 2>&1 | tee {RESULTS_DIR}/train_sft.log
print(f'Baseline SFT wall time minutes: {(time.time() - start) / 60:.2f}')

if RUN_CONTEXT_EXPERIMENT:
    start = time.time()
    !PERSONA=all python3 train_sft_ctx2.py 2>&1 | tee {RESULTS_DIR}/train_{CONTEXT_SUFFIX}_sft.log
    print(f'Context SFT wall time minutes: {(time.time() - start) / 60:.2f}')

if RUN_STATE_CONTEXT_EXPERIMENT:
    start = time.time()
    !PERSONA=all python3 train_sft_state_ctx2.py 2>&1 | tee {RESULTS_DIR}/train_{STATE_CONTEXT_SUFFIX}_sft.log
    print(f'State + context SFT wall time minutes: {(time.time() - start) / 60:.2f}')


## 8. Evaluate Base, Baseline SFT, Context SFT, And State Context SFT

This writes `all_eval_results.json`. The context model is evaluated on `all_ctx2_test.jsonl` because its input format includes previous dialogue turns.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess


def run_eval(test_path, *, base_path='', sft_path='', ctx_model_path='', state_ctx_model_path='', log_name='eval.log'):
    cmd = [
        'python3', 'eval.py',
        '--base', base_path,
        '--sft', sft_path,
        '--ctx-sft', ctx_model_path,
        '--state-ctx-sft', state_ctx_model_path,
        '--test', test_path,
    ]
    print('Running:', ' '.join(part for part in cmd if part))
    env = dict(os.environ)
    env['PERSONA'] = 'all'
    proc = subprocess.run(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(proc.stdout)
    Path(RESULTS_DIR / log_name).write_text(proc.stdout)
    if proc.returncode != 0:
        raise RuntimeError(f'eval.py failed with exit code {proc.returncode}')
    return json.loads(Path('all_eval_results.json').read_text())

baseline_results = run_eval(
    './all_test.jsonl',
    base_path='Qwen/Qwen2.5-1.5B-Instruct',
    sft_path='./all-sft-merged',
    log_name='eval_current.log',
)
combined_results = {
    'base_current_test': baseline_results.get('base'),
    'sft_current_test': baseline_results.get('sft'),
}

if RUN_CONTEXT_EXPERIMENT:
    ctx_model = f'./all-{CONTEXT_SUFFIX}-sft-merged'
    ctx_results = run_eval(
        f'./all_{CONTEXT_SUFFIX}_test.jsonl',
        ctx_model_path=ctx_model if Path(ctx_model).exists() else '',
        log_name=f'eval_{CONTEXT_SUFFIX}.log',
    )
    combined_results[f'{CONTEXT_SUFFIX}_sft_context_test'] = ctx_results.get('ctx_sft')

if RUN_STATE_CONTEXT_EXPERIMENT:
    state_ctx_model = './all-state-ctx2-sft-merged'
    state_ctx_results = run_eval(
        f'./all_{STATE_CONTEXT_SUFFIX}_test.jsonl',
        state_ctx_model_path=state_ctx_model if Path(state_ctx_model).exists() else '',
        log_name=f'eval_{STATE_CONTEXT_SUFFIX}.log',
    )
    combined_results[f'{STATE_CONTEXT_SUFFIX}_sft_context_test'] = state_ctx_results.get('state_ctx_sft')

combined_results = {k: v for k, v in combined_results.items() if v is not None}
Path('all_eval_results.json').write_text(json.dumps(combined_results, indent=2))
shutil.copyfile('all_eval_results.json', RESULTS_DIR / 'all_eval_results.json')
print(json.dumps(combined_results, indent=2)[:3000])


## 9. Display Metrics Table


In [ ]:
import json
from pathlib import Path

results = json.loads(Path('all_eval_results.json').read_text())

print(f"{'model':<30} {'tool_acc':>9} {'imp_acc':>9} {'prec':>7} {'rec':>7} {'f1':>7} {'parse':>7} {'exact':>7} {'detail_f1':>9} {'detail_exact':>12}")
print('-' * 126)
for name, m in results.items():
    print(f"{name:<30} {m['tool_accuracy']:>9.3f} {m['importance_accuracy']:>9.3f} "
          f"{m['action_precision']:>7.3f} {m['action_recall']:>7.3f} "
          f"{m['action_f1']:>7.3f} {m['json_parse_rate']:>7.3f} "
          f"{m.get('exact_match_accuracy', 0.0):>7.3f} "
          f"{m.get('detail_token_f1', 0.0):>9.3f} {m.get('detail_exact_match_accuracy', 0.0):>12.3f}")

for name, m in results.items():
    print('\n' + '=' * 80)
    print(f'Diagnostics for {name}')

    print('\nAction vs no-action confusion:')
    action_conf = m.get('action_vs_no_action_confusion', {})
    print(json.dumps(action_conf.get('matrix', action_conf), indent=2))

    print('\nPer-tool precision / recall / F1:')
    per_tool = m.get('per_tool_precision_recall_f1', {})
    if not per_tool and 'per_tool_f1' in m:
        per_tool = {tool: {'precision': 0.0, 'recall': 0.0, 'f1': f1, 'support': 0}
                    for tool, f1 in m['per_tool_f1'].items()}
    print(f"{'tool':<28} {'support':>7} {'prec':>7} {'rec':>7} {'f1':>7}")
    print('-' * 60)
    for tool, vals in sorted(per_tool.items()):
        print(f"{tool:<28} {vals.get('support', 0):>7} "
              f"{vals.get('precision', 0.0):>7.3f} {vals.get('recall', 0.0):>7.3f} "
              f"{vals.get('f1', 0.0):>7.3f}")

    print('\nWeak-class recall:')
    print(f"{'class':<28} {'support':>7} {'correct':>7} {'recall':>7}")
    print('-' * 60)
    for cls, vals in sorted(m.get('weak_class_recall', {}).items()):
        print(f"{cls:<28} {vals.get('support', 0):>7} {vals.get('correct', 0):>7} "
              f"{vals.get('recall', 0.0):>7.3f}")

    print('\nTool confusion matrix:')
    print(json.dumps(m.get('tool_confusion_matrix', {}), indent=2))

    print('\nImportance confusion matrix:')
    print(json.dumps(m.get('importance_confusion_matrix', {}), indent=2))


## 10. Optional: Compare Against Previous Results

If you have an older eval JSON, upload it and set `OLD_RESULTS_PATH` below.


In [ ]:
# Optional upload of previous eval results
# from google.colab import files
# files.upload()

OLD_RESULTS_PATH = None  # example: 'old_all_eval_results.json'

if OLD_RESULTS_PATH:
    old = json.loads(Path(OLD_RESULTS_PATH).read_text())
    new = json.loads(Path('all_eval_results.json').read_text())
    print(f"{'model':<8} {'metric':<22} {'old':>8} {'new':>8} {'delta':>8}")
    print('-' * 62)
    for model in sorted(set(old) & set(new)):
        for metric in ['tool_accuracy', 'importance_accuracy', 'exact_match_accuracy', 'action_precision', 'action_recall', 'action_f1', 'json_parse_rate']:
            o = old[model][metric]
            n = new[model][metric]
            print(f'{model:<8} {metric:<22} {o:>8.3f} {n:>8.3f} {n-o:>8.3f}')
else:
    print('Set OLD_RESULTS_PATH to compare with a previous eval JSON.')


## 11. Save And Download Results

This packages metrics, logs, and model folders. The zip can be large because it includes merged checkpoints.


In [ ]:
import shutil
from pathlib import Path

# Save compact metadata/results first.
for f in [
    'all_eval_results.json', 'all_test.jsonl', 'all_sft.jsonl', 
    f'all_{CONTEXT_SUFFIX}_test.jsonl', f'all_{CONTEXT_SUFFIX}_sft.jsonl',
    f'all_{CONTEXT_SUFFIX}_stats.json',
    f'all_{STATE_CONTEXT_SUFFIX}_test.jsonl', f'all_{STATE_CONTEXT_SUFFIX}_sft.jsonl',
    f'all_{STATE_CONTEXT_SUFFIX}_stats.json',
]:
    if Path(f).exists():
        shutil.copy(f, RESULTS_DIR / f)

# Optional: include merged model folders in the archive.
INCLUDE_MODELS_IN_ZIP = False

if INCLUDE_MODELS_IN_ZIP:
    for folder in ['all-sft-merged', f'all-{CONTEXT_SUFFIX}-sft-merged', 'all-state-ctx2-sft-merged']:
        if Path(folder).exists():
            shutil.copytree(folder, RESULTS_DIR / folder, dirs_exist_ok=True)

zip_path = shutil.make_archive(str(RESULTS_DIR), 'zip', RESULTS_DIR)
print('Saved:', zip_path)

# Download compact results archive.
from google.colab import files
files.download(zip_path)


## 12. Optional: Save Model Folders To Drive

Run this if you want to keep the trained checkpoints in Drive for later Ollama/GGUF conversion.


In [ ]:
from pathlib import Path
import shutil, time

MODEL_SAVE_DIR = Path('/content/drive/MyDrive/CSE534_models') / RUN_ID
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

for folder in ['all-sft-merged', f'all-{CONTEXT_SUFFIX}-sft-merged', 'all-state-ctx2-sft-merged']:
    if Path(folder).exists():
        shutil.copytree(folder, MODEL_SAVE_DIR / folder, dirs_exist_ok=True)
        print('Saved', folder, 'to', MODEL_SAVE_DIR / folder)
